# Lesson 15 — Presolve and FBBT

## Learning objectives

1. Recognize the standard presolve transformations.
2. Implement feasibility-based bound tightening (FBBT) by hand.
3. Understand probing and its trade-offs.
4. Use `discopt`'s presolve flags to debug bad models.

## 1. Why presolve matters

Presolve runs *before* B&B and aims to reduce problem size, tighten bounds, and detect infeasibility. On real MILPs, presolve commonly reduces variable count 30–60% and constraint count 20–50% {cite:p}`Savelsbergh1994,Achterberg2007`. Time spent in presolve is almost always recovered.

## 2. Standard transformations

| Transformation | Example |
|----------------|---------|
| Singleton row | $x_j \ge 5$ from a single-variable constraint → tighten bound |
| Empty column | Variable with 0 coefficients → fix to bound that helps objective |
| Implied bound | $x_1 + x_2 + x_3 = 1, x_i \in [0,1]$ → already tight |
| Coefficient strengthening | Loose big-M → tighten via implied bounds |
| Aggregation | $x_1 = 2 x_2 + 3$ → eliminate $x_1$ |
| Probing | Set $x_j = 0$, propagate; if infeasible, fix $x_j = 1$ |

See {cite:p}`Savelsbergh1994` and {cite:p}`Belotti2009` for the systematic catalogue.

## 3. Feasibility-based bound tightening (FBBT)

Given a constraint $\sum_j a_j x_j \le b$ with $x_j \in [\ell_j, u_j]$, the implied bound on each $x_k$ is

$$x_k \le \frac{b - \sum_{j \ne k, a_j > 0} a_j \ell_j - \sum_{j \ne k, a_j < 0} a_j u_j}{a_k} \quad \text{(if } a_k > 0\text{)}.$$

(Mirror for $a_k < 0$ and for $\ge$ constraints.) Apply to every variable in every constraint; iterate to a fixed point. {cite:p}`Belotti2009` formalize this for the MINLP setting.

## 4. Probing

Pick a binary $x_j$. Tentatively set $x_j = 0$, run FBBT. If infeasible, fix $x_j = 1$. Repeat for $x_j = 1$. Either reduces variables or generates *implied bounds* and *cliques* ($x_j = 1 \Rightarrow x_k = 0$). Quadratic in the number of binaries; usually limited to a budget {cite:p}`Achterberg2007`.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do

# Problem with redundant variables and slack
m = do.Model()
x = m.add_variables(5, lb=0, ub=10)
m.add_constraint(x[0] + x[1] + x[2] <= 100)        # redundant
m.add_constraint(x[3] - 2*x[4] == 5)               # use to eliminate x[3]
m.add_constraint(x[0] + 2*x[1] >= 8)               # tightens
m.minimize(x.sum())

stats = m.presolve()
print("vars before/after :", stats.vars_before, "->", stats.vars_after)
print("rows before/after :", stats.rows_before, "->", stats.rows_after)
print("bounds tightened  :", stats.bounds_tightened)


## 5. FBBT in MINLP

For nonlinear constraints, FBBT uses **interval arithmetic** {cite:p}`Moore1966,Neumaier1990`. Given $z = x \cdot y$ with $x \in [a, b], y \in [c, d]$, the implied bound is $z \in [\min(ac, ad, bc, bd), \max(...)]$.

This yields tight enclosures for many problems but can be loose for high-rank monomials. **Optimization-based BT (OBBT)** instead solves a small auxiliary LP/NLP to compute each bound — much tighter, much slower.

## 6. Effects on B&B

Tighter bounds → tighter LP relaxation → more pruning → smaller tree. Plus tighter bounds → tighter big-M and McCormick relaxations → tighter convex envelopes.

This is why "pre-modelling" (giving good initial bounds in your formulation) is usually the highest-leverage thing you can do for solver performance.

## References
{cite:p}`Savelsbergh1994` (LP/MILP), {cite:p}`Belotti2009` (MINLP), {cite:p}`Achterberg2007`.